Step 1 — Mount Drive and set paths:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/CVE_Project'
RAW  = f'{BASE}/raw_data'
os.makedirs(RAW, exist_ok=True)
print("Folders ready")

Mounted at /content/drive
Folders ready


Step 2 — Install nvdlib

In [ ]:
!pip install nvdlib pandas

Step 3 —  Fetch with 120-day chunks

In [ ]:
import nvdlib
import pandas as pd
from datetime import datetime, timedelta

API_KEY = "89c76cde-5676-4bb9-94ac-690792a7007b"

def score_to_label(score):
    if score >= 9.0:   return "Critical"
    elif score >= 7.0: return "High"
    elif score >= 4.0: return "Medium"
    else:              return "Low"

def get_chunks(start_year, end_year, chunk_days=100):
    """Split date range into chunks of max 100 days (safely under 120 limit)"""
    chunks = []
    current = datetime(start_year, 1, 1)
    end     = datetime(end_year, 12, 31)
    while current < end:
        chunk_end = min(current + timedelta(days=chunk_days), end)
        # nvdlib expects format: '2021-09-08 00:00'
        chunks.append((
            current.strftime('%Y-%m-%d 00:00'),
            chunk_end.strftime('%Y-%m-%d 23:59')
        ))
        current = chunk_end + timedelta(days=1)
    return chunks

chunks = get_chunks(2019, 2023)
print(f"Total chunks to fetch: {len(chunks)}")
for c in chunks[:3]:
    print(c)   # preview first 3

Total chunks to fetch: 19
('2019-01-01 00:00', '2019-04-11 23:59')
('2019-04-12 00:00', '2019-07-21 23:59')
('2019-07-22 00:00', '2019-10-30 23:59')


Step 4 — Run the fetch

In [ ]:
all_cves = []

for i, (start, end) in enumerate(chunks):
    print(f"Chunk {i+1}/{len(chunks)}: {start} → {end}")
    try:
        results = nvdlib.searchCVE(
            pubStartDate = start,
            pubEndDate   = end,
            key          = API_KEY,
            delay        = 0.6
        )
        count = 0
        for cve in results:
            try:
                # Get English description
                desc = ""
                for d in cve.descriptions:
                    if d.lang == "en":
                        desc = d.value
                        break

                if not desc or "** REJECT **" in desc or len(desc.split()) < 10:
                    continue

                # Get CVSS v3 score
                score = None
                vector = complex_ = privs = ui = scope = ""

                if hasattr(cve.metrics, 'cvssMetricV31') and cve.metrics.cvssMetricV31:
                    c = cve.metrics.cvssMetricV31[0].cvssData
                    score   = c.baseScore
                    vector  = c.attackVector
                    complex_= c.attackComplexity
                    privs   = c.privilegesRequired
                    ui      = c.userInteraction
                    scope   = c.scope
                elif hasattr(cve.metrics, 'cvssMetricV30') and cve.metrics.cvssMetricV30:
                    c = cve.metrics.cvssMetricV30[0].cvssData
                    score   = c.baseScore
                    vector  = c.attackVector
                    complex_= c.attackComplexity
                    privs   = c.privilegesRequired
                    ui      = c.userInteraction
                    scope   = c.scope

                if score is None:
                    continue

                all_cves.append({
                    "cve_id":              cve.id,
                    "description":         desc,
                    "cvss_score":          score,
                    "cvss_label":          score_to_label(score),
                    "attack_vector":       vector,
                    "attack_complexity":   complex_,
                    "privileges_required": privs,
                    "user_interaction":    ui,
                    "scope":               scope
                })
                count += 1

            except Exception:
                continue

        print(f"  Got {count} CVEs — running total: {len(all_cves)}")

    except Exception as e:
        print(f"  ERROR on chunk {i+1}: {e}")
        continue

print(f"\nDone. Total CVEs collected: {len(all_cves)}")

Chunk 1/19: 2019-01-01 00:00 → 2019-04-11 23:59
  Got 3957 CVEs — running total: 3957
Chunk 2/19: 2019-04-12 00:00 → 2019-07-21 23:59
  Got 4222 CVEs — running total: 8179
Chunk 3/19: 2019-07-22 00:00 → 2019-10-30 23:59
  Got 5419 CVEs — running total: 13598
Chunk 4/19: 2019-10-31 00:00 → 2020-02-08 23:59
  Got 5093 CVEs — running total: 18691
Chunk 5/19: 2020-02-09 00:00 → 2020-05-19 23:59
  Got 5569 CVEs — running total: 24260
Chunk 6/19: 2020-05-20 00:00 → 2020-08-28 23:59
  Got 4603 CVEs — running total: 28863
Chunk 7/19: 2020-08-29 00:00 → 2020-12-07 23:59
  Got 4535 CVEs — running total: 33398
Chunk 8/19: 2020-12-08 00:00 → 2021-03-18 23:59
  Got 4901 CVEs — running total: 38299
Chunk 9/19: 2021-03-19 00:00 → 2021-06-27 23:59
  Got 5163 CVEs — running total: 43462
Chunk 10/19: 2021-06-28 00:00 → 2021-10-06 23:59
  Got 5815 CVEs — running total: 49277
Chunk 11/19: 2021-10-07 00:00 → 2022-01-15 23:59
  Got 5399 CVEs — running total: 54676
Chunk 12/19: 2022-01-16 00:00 → 2022-04-26 

Step 5 — Save to Drive

---

A few things to know before running:

The fetch in Cell 3 will take roughly 15–30 minutes because it is pulling thousands of records. Don't close the browser tab while it runs. If it disconnects halfway, nvdlib doesn't support resuming, so you'd need to re-run — but that's fine since it's just an API call.

When Cell 4 prints the label counts, you're looking for something roughly like this — if Critical is more than 40% of your data, something went wrong with the score mapping:
```
High        6200
Medium      5100
Critical    2800
Low          900

In [ ]:
df = pd.DataFrame(all_cves)
print(df.shape)
print("\nLabel distribution:")
print(df["cvss_label"].value_counts())

df.to_csv(f'{RAW}/cves_raw.csv', index=False)
print(f"\nSaved to {RAW}/cves_raw.csv")

(105361, 9)

Label distribution:
cvss_label
Medium      46397
High        40454
Critical    14807
Low          3703
Name: count, dtype: int64

Saved to /content/drive/MyDrive/CVE_Project/raw_data/cves_raw.csv


In [ ]:
import pandas as pd

df = pd.read_csv(f'{RAW}/cves_raw.csv')
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nSample row:")
print(df.iloc[0])
print(f"\nAny nulls:\n{df.isnull().sum()}")

Rows: 105361
Columns: ['cve_id', 'description', 'cvss_score', 'cvss_label', 'attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'scope']

Sample row:
cve_id                                                     CVE-2019-3494
description            Simply-Blog through 2019-01-01 has SQL Injecti...
cvss_score                                                           7.5
cvss_label                                                          High
attack_vector                                                    NETWORK
attack_complexity                                                    LOW
privileges_required                                                 NONE
user_interaction                                                    NONE
scope                                                          UNCHANGED
Name: 0, dtype: object

Any nulls:
cve_id                 0
description            0
cvss_score             0
cvss_label             0
attack_vector          0
atta